In [3]:
import json
import os

# Load Kaggle API credentials from kaggle_key.json
with open('kaggle_key.json', 'r') as f:
    kaggle_credentials = json.load(f)

# Set Kaggle API credentials as environment variables
os.environ['KAGGLE_USERNAME'] = kaggle_credentials['username']
os.environ['KAGGLE_KEY'] = kaggle_credentials['key']

print("Kaggle API credentials loaded successfully!")
print(f"Username: {kaggle_credentials['username']}")
print("Key: ****" + kaggle_credentials['key'][-4:])  # Show only last 4 characters for security

Kaggle API credentials loaded successfully!
Username: cubbic
Key: ****f9db


In [4]:

os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

import keras_hub
# Load the base model
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_4b_text")

# Enable LoRA with the same rank as during training
gemma_lm.backbone.enable_lora(rank=4) # type: ignore

# Load the trained LoRA weights
gemma_lm.backbone.load_lora_weights("trained_gemma_4b_disaster_lora.lora.h5") # type: ignore

# Compile with the same sampler for inference
sampler = keras_hub.samplers.GreedySampler()
gemma_lm.compile(sampler=sampler)




2025-08-06 22:35:42.657313: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1754512542.658455    6572 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1652 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:31:00.0, compute capability: 8.9
normalizer.cc(51) LOG(INFO) precompiled_charsmap is empty. use identity normalization.


In [5]:
import pandas as pd

test = pd.read_csv("data/test.csv")

In [ ]:
template = "Tweet: {tweet}\nIs this about a real disaster? Answer Yes or No\nAnswer:"
print(len(test))

submission = pd.DataFrame(columns=['id', 'target'])
submission['id'] = test['id']

predictions = []
for i, row in test.iterrows():
    prompt = template.format(tweet=row['text'])
    response = gemma_lm.generate(prompt)
    last_3_chars = response[-3:]
    answer = -1
    if("answer:yes" in response.lower()):
        answer = 1
    elif("answer:no" in response.lower()):
        answer = 0
    predictions.append(answer)

    if(answer == -1):
        print(f"Row {i}/{len(test)}: No answer found in response: {response}")
    
    print(f"Row {i}/{len(test)}: {answer}")

submission['target'] = predictions

submission.to_csv('data/submission_gemma_4b.csv', index=False)

3263
Row 0/3263: No answer found in response: Tweet: Just happened a terrible car crash
Is this about a real disaster? Answer Yes or No
Answer:No
Row 0/3263: -1
Row 1/3263: No answer found in response: Tweet: Heard about #earthquake is different cities, stay safe everyone.
Is this about a real disaster? Answer Yes or No
Answer:Yes
Row 1/3263: -1
Row 2/3263: No answer found in response: Tweet: there is a forest fire at spot pond, geese are fleeing across the street, I cannot save them all
Is this about a real disaster? Answer Yes or No
Answer:Yes
Row 2/3263: -1
Row 3/3263: No answer found in response: Tweet: Apocalypse lighting. #Spokane #wildfires
Is this about a real disaster? Answer Yes or No
Answer:Yes
Row 3/3263: -1
Row 4/3263: No answer found in response: Tweet: Typhoon Soudelor kills 28 in China and Taiwan
Is this about a real disaster? Answer Yes or No
Answer:Yes
Row 4/3263: -1
Row 5/3263: No answer found in response: Tweet: We're shaking...It's an earthquake
Is this about a rea

KeyboardInterrupt: 